# Data processing

As **input**, this script takes raw timeseries from the stations and models, saved in .nc files:
- Separately for each model
- Exactly 10 years for models, up to 30 years for observations
- Precipitation and temperature
- Hourly resolution
- Only station locations, i.e. for the models it's not gridded data, but timeseries from 277 grid cells that are nearest neighbors of the stations
- The files were processed to exclude obivous outliers, e.g. timesteps with more than 100mm/h averaged over all stations (i.e., large scale). Similar outliers were found and removed from station data (precipitation over ~200 mm/h). It is possible that some outliers remain.

As **output**, this script produces two types of auxiliary .nc files where intermediate results are stored:
- **extended hourly timeseries**: dataset with hourly frequency which contains:
    - hourly precipitation timeseries
    - hourly temperature timeseries
    - mean daily temperature timeseries broadcasted to an hourly frequency
    - total daily precipitation timeseries broadcasted to hourly frequency
- **extended daily timeseries**: dataset which contains the following timeseries with daily frequency:
    - 'mean_daily_tas' (average temperature over 24 hours)
    - 'pr' (total daily precipitation sum over 24 hours)
    - 'wet_hour_count' (number of wet hours within the day, used to determine wet hour freqnency later in the analysis)
    - 'wet_hour_mean_intensity' (how much precipitation falls within an hour if the hour is wet?
        Note: this is calculated separately for each day here; nan if there are zero wet hours in a day)
    - 'wet_hour_max_intensity' (what's the most intense precipitation that falls in that day?
        Note: this in NOT equivalent to high percentiles as this is calculated separately for each day;
        nan if there are zero wet hours in a day)
    - 'pr_onset_time' (time of precipitation onset, i.e. time in UTC when the first wet hour occurs.
        This is calculated only for days with at least one wet hour)

The following .nc files are produced from the extended timeseries and serve as the main output of this script, used in later processing:
- **quantiles_DatasetName_TimePeriod_hdmean_Season.nc**: "hdmean" stands for "hourly precipitation, daily mean tempearture". 
    The files contain hourly precipitation quantiles, count of wet hours, and count of all hours in each temperature bin.
- **quantiles_DatasetName_TimePeriod_ddmean_Season.nc**: "ddmean" stands for "daily precipitation, daily mean tempearture". 
    The files contain daily precipitation quantiles, count of wet days, and count of all days in each temperature bin.
- **various_daily_stats_DatasetName_TimePeriod_dmean_Season.nc**: "dmean" stands for "daily mean tempearture". 
    The files contain mean daily precipitation, count of days at each temperature, mean and maximum of wet hour intensities,
        time of onset of precipitation, as well as all these aforementioned characteristics separated by 
        various ranges of mean daily precipitation (0.1-1, 1-2.5, 25.-5, 5-10, 10+ mm/d).
        
One file is produced per time period (evaluation/historical/rcp), per season (DJF/MAM/JJA/SON/all year), per model.
Subsequently, ensemble files are created, containing all the models per time period and per season.

**Note**: currently, this script creates 185 GB of additional processed data.

This could be reduced to ~100 GB with relatively simple code changes: instead of saving "extended_daily_measures_\*" and "extended_hourly_measures_\*" separately for all seasons (allseasons) and each individual season (DJF, MAM, JJA, SON), this could be done for all seasons once, and season selection would follow afterwards (function make_all_datasets_from_hourly_data())

In [1]:
import concurrent.futures
import glob
import os

import cftime
import numpy as np
import pandas as pd
import xarray as xr
from flox.xarray import xarray_reduce

# Definitions

In [ ]:
# to run the script, change this based on where you saved the data
input_folder = "."

# paths to data: this code cell has to be adjusted to where your files are saved if you want to reuse the script.
# parent folder where all raw station and model data is saved
folder_raw_data = f"{input_folder}/raw_data"

# path to the folder where processed files will be saved
folder_processed_data = f"{input_folder}/processed_data"

# flag to save files: set to false for development/debugging reasons
savefiles = True

if savefiles:
    # if the output folder doesn't exist yet, create it
    os.makedirs(folder_processed_data, exist_ok=True)

# cores to use in parallelized processing
workers = 32

In [4]:
# station data
filepath_geosphere = f"{folder_raw_data}/station_timeseries_geosphere_1990-2020.nc"
ds_geosphere = xr.open_dataset(filepath_geosphere)

# metadata to be added to the processed files
metadata = ds_geosphere[["lat", "lon", "elevation", "station_id"]]

In [5]:
# all evaluation files: km-scale + driving (coarse)
files_kmscale_eval = sorted(
    glob.glob(f"{folder_raw_data}/ALP-3/evaluation/*/*1hr_complete.nc")
)
files_driving_eval = sorted(
    glob.glob(f"{folder_raw_data}/EUR-12/evaluation/*/*1hr_complete.nc")
)

# all historical and RCP8.5 files: km-scale + driving (coarse)
# include CLMcom-ETH-GAR/REU from eval in hist, and include pgw in rcp:
# the eval/PGW pairs of these two models imitate hist+rcp scenarios in other models
files_kmscale_hist = sorted(
    glob.glob(f"{folder_raw_data}/ALP-3/historical/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/ALP-3/evaluation/CLMcom-ETH-*/*1hr_complete.nc")
)
files_driving_hist = sorted(
    glob.glob(f"{folder_raw_data}/EUR-12/historical/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/EUR-12/evaluation/CLMcom-ETH-*/*1hr_complete.nc")
)

files_kmscale_rcp = sorted(
    glob.glob(f"{folder_raw_data}/ALP-3/rcp85/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/ALP-3/PGW-MPI-rcp85/*/*1hr_complete.nc")
)
files_driving_rcp = sorted(
    glob.glob(f"{folder_raw_data}/EUR-12/rcp85/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/EUR-12/PGW-MPI-rcp85/*/*1hr_complete.nc")
)

In [6]:
# bins for grouping data: calculate overlapping binned series to get smoother data
# it is necessary to have two arrays with step size 1 ("bin width") for some intermediate processing, the final array serves as an index
t_bins_1 = np.arange(-15, 36, 1)
t_bins_2 = np.arange(-14.5, 36.5, 1)
t_bins = np.ravel([t_bins_1[:-1], t_bins_2[:-1]], order="F")

# define thresholds for wet/dry days
WET_HOUR_INTENSITY = 0.1  # mm/h
WET_DAY_INTENSITY = 1  # mm/d

In [7]:
# naming used for various ranges of mean daily precipitation
dict_pr_ranges = {
    "all": (0.1, 999),      # all days with at least one wet hour
    "wet": (1, 999),        # all wet days (at least 1mm/day)
    "0p1_1p0": (0.1, 1),
    "1p0_2p5": (1, 2.5),
    "2p5_5p0": (2.5, 5),
    "05_10": (5, 10),
    "10_plus": (10, 999),
}

In [8]:
# season names and the corresponding months
seasons = ["DJF", "MAM", "JJA", "SON", "allseasons"]
months = [[12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11], None]

# Functions

In [9]:
def bin_values(ds, var_grouped, groupby_var, func, quantiles=None):
    """
    This function is where the grouping by tmeperature happens. 
    The input data array contains 'time' as a dimension; the output contains 'temperature bin'.
    The data is grouped by intervals, not by discrete values

    Args:
        ds (xr.Dataset): A dataset with hourly or daily time resolution. Contains 'time' as a dimension.
        var_grouped (str): The variable to be grouped. In our case this is mostly some characteristic of precipitation, e.g. hourly intensity.
        groupby_var (str): Variable to group by. In our case this is mean daily temperature ('mean_daiy_tas').
        func (str): Function to apply to the groups. This function is applied to each bin. One of:
            'mean': takes an average over a given bin. Used to calculate mean daily precipitation, mean and maximum hourly intensities
            'count': counts the number of items in a given bin. Used to determine the frequency of all/wet hours/days.
            'nanquantile': calculates the specified quantiles of a given bin. Used to calculated all-hour percentiles.
        quantiles (list, optional): List of floats between 0-1. These represent the quantiles to be calculated. 
            If None, no quantiles are calculated. Defaults to None.

    Returns:
        groups_concat (xr.DataArray): a matrix of a statistic as a function of temperature. The dimensions are temperature_bin, station_name, and for model data also model
    """

    # prepare dataset for grouping: set NaNs at all places where one of the variables is NaN
    ds_to_group = ds.where((ds[var_grouped].notnull() & ds[groupby_var].notnull()))

    # same arguments for all grouping
    kwargs = {
        "dim": "time",         # grouping is done along this dimension (it does not appear in the resulting dataset)
        "isbin": True,         # the data is grouped by intervals, not by discrete values
        "fill_value": np.nan,  # fill value for groups with no members
        "func": func,          # function to be performed on each group: mean, count, or nanquantile
    }

    # add another kwarg when calculating quantiles: not applicable for mean, max, count, etc
    if quantiles is not None:
        kwargs["q"] = quantiles

    # group the first variable (mostly something to do with precipitation) by the second one (mean daily temperature)
    # group by first array of temperature bins: 0-1, 1-2, 2-3, ...
    group_1 = xarray_reduce(
        ds_to_group[var_grouped],
        ds_to_group[groupby_var],
        expected_groups=pd.IntervalIndex.from_breaks(t_bins_1),
        **kwargs,
    )
    # group by second array of temperature bins: 0.5-1.5, 1.5-2.5, 2.5-3.5, ...
    group_2 = xarray_reduce(
        ds_to_group[var_grouped],
        ds_to_group[groupby_var],
        expected_groups=pd.IntervalIndex.from_breaks(t_bins_2),
        **kwargs,
    )

    # concatenate the two groups, sort by increasing temperature (so that the two groups alternate: 0-1, 0.5-1.5, 1-2, 1.5-2.5, ...)
    groups_concat = xr.concat([group_1, group_2], dim=f"{groupby_var}_bins").sortby(f"{groupby_var}_bins")

    # rename the grouping variables so that it's consistent among all variables
    groups_concat = groups_concat.rename({f"{groupby_var}_bins": "temperature_bin"})
    
    return groups_concat

In [10]:
def get_quantiles_and_counts(ds, groupby_var='mean_daily_tas', wet_intensity=WET_HOUR_INTENSITY):
    """
    This function calculates the all-hour/all-day quantiles, and the count of all/wet days and hours.
    These are calculated for each temperature bin separately.

    Args:
        ds (xr.Dataset): A dataset with hourly or daily time resolution. 
        groupby_var (str, optional): Variable that the array is grouped by. Defaults to 'mean_daily_tas'.
        wet_intensity (float, optional): A threshold above which the hour/day is considered to be wet. Defaults to WET_HOUR_INTENSITY (global constant).

    Returns:
        quantiles_all (xr.DataArray): high percentiles of hourly/daily precipitation vs. mean daily temperature
        count_all (xr.DataArray): count of all hours/days
        count_wet (xr.DataArray): count of wet hours/days
    """

    # get all/wet hours/days
    ds_all = ds
    ds_wet = ds.where(ds.pr >= wet_intensity)

    # calculate quantiles for each bin (all hours/days, not only wet ones), 1 is basically max
    quantiles = [0.9, 0.95, 0.99, 0.999, 1]
    quantiles_all = bin_values(ds_all, "pr", groupby_var, "nanquantile", quantiles=quantiles)

    # get count of all values, wet values, dry values in each bin
    count_all = bin_values(ds_all, "pr", groupby_var, "count")
    count_wet = bin_values(ds_wet, "pr", groupby_var, "count")

    return quantiles_all, count_all, count_wet

In [11]:
def get_daily_stats(ds_hourly):
    """
    This function calculates certain precipitation stats for EVERY SINGLE DAY, before any grouping occurs.

    We also take the hourly timeseries and extend them by two mean daily temperature and daily precipitation sum,
    which are reindexed from daily to hourly frequency:

    All these measures serve for later grouping by temperature and daily precipitation sums

    Args:
        ds_hourly (xr.Dataset): input dataset which contains hourly precipitation and temperature timeseries plus some metadata.

    Returns:
        ds_hourly_extended (xr.Dataset): dataset with hourly frequency which contains:
            - hourly precipitation timeseries
            - hourly temperature timeseries
            - mean daily temperature timeseries broadcasted to an hourly frequency
            - total daily precipitation timeseries broadcasted to hourly frequency
        ds_daily_extended (xr.Dataset): dataset which daily frequency which contains the following timeseries:
            - 'mean_daily_tas' (average temperature over 24 hours)
            - 'pr' (total daily precipitation sum over 24 hours)
            - 'wet_hour_count' (number of wet hours within the day, used to determine wet hour freqnency later in the analysis)
            - 'wet_hour_mean_intensity' (how much precipitation falls within an hour if the hour is wet?
                Note: this is calculated separately for each day here; nan if there are zero wet hours in a day)
            - 'wet_hour_max_intensity' (what's the most intense precipitation that falls in that day?
                Note: this in NOT equivalent to high percentiles as this is calculated separately for each day;
                nan if there are zero wet hours in a day)
            - 'pr_onset_time' (time of precipitation onset, i.e. time in UTC when the first wet hour occurs.
                This is calculated only for days with at least one wet hour)
    """
    
    # if needed, convert 360 day calendar to 365 day calendar: https://docs.xarray.dev/en/stable/generated/xarray.Dataset.convert_calendar.html
    # this is relevant for only a few models
    if type(ds_hourly.time[0].item()) is cftime._cftime.Datetime360Day:
        ds_hourly = ds_hourly.convert_calendar(calendar="gregorian", align_on="year")
    else:
        pass

    if "station_name" not in ds_hourly.dims:
        ds_hourly = ds_hourly.rename({"stations": "station_name"})

    # get mean and max daily temperature: skipna False ensures that days which have at least one NaN are filled with zeros: 
    # otherwise e.g. a sum of 24 NaNs is 0 which I don't want
    mean_daily_tas = ds_hourly.tas.groupby(ds_hourly.time.dt.date).mean(skipna=False)

    # get total daily precipitation
    total_daily_pr = ds_hourly.pr.groupby(ds_hourly.time.dt.date).sum(skipna=False)

    # get a mask of wet hours (boolean)
    ds_hourly_wet = (ds_hourly.pr >= WET_HOUR_INTENSITY)
    # group wet hours by date
    pr_hourly_grouped = ds_hourly.pr.where(ds_hourly_wet).groupby(ds_hourly.time.dt.date)

    # get count of wet hours (sum of "True", i.e. 1, for every day)
    wet_hour_count = ds_hourly_wet.groupby(ds_hourly.time.dt.date).sum(skipna=True)

    # get intensity on days with wet hours: mean and maximum of EVERY DAY
    wet_hour_mean_intensity = pr_hourly_grouped.mean(skipna=True)
    wet_hour_max_intensity = pr_hourly_grouped.max(skipna=True)

    # get the first hour when precipitation occurs, add that to the daily data
    pr_onset_time = (ds_hourly.time.dt.hour).where(ds_hourly_wet).groupby(ds_hourly.time.dt.date).first()

    # create a dataset from individual variables
    ds_daily_extended = xr.Dataset(
        data_vars={
            "mean_daily_tas": mean_daily_tas,
            "pr": total_daily_pr,
            "wet_hour_count": wet_hour_count,
            "wet_hour_mean_intensity": wet_hour_mean_intensity,
            "wet_hour_max_intensity": wet_hour_max_intensity,
            "pr_onset_time": pr_onset_time,
        }
    )

    # change the name "date" to "time" in the daily timeseires for late processing
    try:
        ds_daily_extended["date"] = pd.to_datetime(ds_daily_extended.date)
        ds_daily_extended = ds_daily_extended.rename({"date": "time"})
    except AttributeError:
        ds_daily_extended["floor"] = pd.to_datetime(ds_daily_extended.floor)
        ds_daily_extended = ds_daily_extended.rename({"floor": "time"})

    # extend hourly statistics: add the required daily values to hourly data, this will serve for later grouping of hourly data by daily values
    ds_hourly_extended = ds_hourly
    ds_hourly_extended["mean_daily_tas"] = ds_daily_extended["mean_daily_tas"].reindex_like(ds_hourly_extended, method="ffill")
    ds_hourly_extended["pr_daily"] = ds_daily_extended["pr"].reindex_like(ds_hourly_extended, method="ffill")
    
    return ds_hourly_extended, ds_daily_extended

In [12]:
def make_quantiles_file(ds_timeseries, metadata, groupby_var='mean_daily_tas', wet_intensity=WET_HOUR_INTENSITY, description=None, dataset_name=None):
    """
    The function takes hourly/daily timeseries as an input and creates a file with hourly/daily precipitation quantiles and a count of all/wet days/hours

    Args:
        ds_timeseries (xr.Dataset): input dataset which contains hourly or daily precipitation and temperature timeseries.
        metadata (xr.Dataset): metadata about a location: elevation, latitude, longitude, station ID
        groupby_var (str, optional): Variable that the array is grouped by. Defaults to 'mean_daily_tas'.
        wet_intensity (float, optional): A threshold above which the hour/day is considered to be wet. Defaults to WET_HOUR_INTENSITY (global constant).
        description (str, optional): Metadata to be added to dataset attributes. Defaults to None.
        quantiles (list, optional): List of floats between 0-1. These represent the quantiles to be calculated. If None, no quantiles are calculated. Defaults to None.
        dataset_name (str): One of 'kmscale', 'driving', 'GeoSphere'. Explains what type of dataset is being processed and is used in the output file name.

    Returns:
        ds_groupby_tas (xr.Dataset): resulting dataset with quantities grouped by mean daily temperature
    """

    # calculate quantiles, mean and max precipitation, counts, etc
    quantiles_all, count_all, count_wet = get_quantiles_and_counts(ds_timeseries, groupby_var=groupby_var, wet_intensity=wet_intensity)

    # make a dataset with all the calculated variables
    ds_groupby_tas = xr.Dataset(dict(
        prcp_percentiles_all=quantiles_all,
        count_all=count_all,
        count_wet=count_wet,
        elevation=(["station_name"], metadata['elevation'].values),
        station_id=(["station_name"], metadata['station_id'].values),
        lon=(["station_name"], metadata['lon'].values),
        lat=(["station_name"], metadata['lat'].values)
        ),

    # add attributes in file info
    attrs=dict(
        desctiption=description,
        dataset_name=dataset_name,
        wet_intensity=wet_intensity,
        )
    )
    # change temperature bins to values instead of intervals
    ds_groupby_tas["temperature_bin"] = [interval.left for interval in ds_groupby_tas.temperature_bin.values]

    return ds_groupby_tas

In [13]:
def make_complete_daily_file(ds_daily_timeseries, metadata, groupby_var='mean_daily_tas', dataset_name=None):
    """
    This function takes daily timeseries of various precipitation statistics as an input and produces a dataset where these statistics
    are stored as a function of temperature

    Args:
        ds_daily_timeseries (xr.Dataset): daily timeseries extended by derived quantities
        metadata (xr.Dataset): metadata about a location: elevation, latitude, longitude, station ID
        groupby_var (str, optional): Variable that the array is grouped by. Defaults to 'mean_daily_tas'.
        dataset_name (str): One of 'kmscale', 'driving', 'GeoSphere'. Explains what type of dataset is being processed and is used in the output file name.

    Returns:
        ds_groupby_tas (xr.Dataset): resulting dataset with ~40 quantities grouped by mean daily temperature
    """

    # create a dataset to hold the results
    ds_groupby_tas = xr.Dataset(
        dict(
            # what's the mean daily precipitation? - average over all days (even dry) at a given temperature
            mean_daily_precipitation=bin_values(ds_daily_timeseries, 'pr', groupby_var, 'mean'),
            # how many times does this bin occur? - get bin counts, including dry days
            daily_count=bin_values(ds_daily_timeseries, 'pr', groupby_var, 'count'),
            # the usual stats present in each dataset
            elevation=(["station_name"], metadata["elevation"].values),
            station_id=(["station_name"], metadata["station_id"].values),
            lon=(["station_name"], metadata["lon"].values),
            lat=(["station_name"], metadata["lat"].values),
        ),
        # add attributes in file info
        attrs=dict(
            desctiption="Various daily stats grouped by temperature bins. Bins of 1°C, overlapping by 0.5°C.",
            dataset_name=dataset_name,
            wet_hour_intensity=WET_HOUR_INTENSITY,
            wet_day_intensity=WET_DAY_INTENSITY,
        ),
    )

    for range_name, (range_min, range_max) in zip(dict_pr_ranges.keys(), dict_pr_ranges.values()):
        # identify days which fall within a given daily precipitation sum (e.g. 0.1-1 mm/day, 1-2.5mm/day, ...)
        days_within_range = ds_daily_timeseries.where((ds_daily_timeseries.pr >= range_min) & (ds_daily_timeseries.pr < range_max))

        # calculate statistics for that precipitation range and add then to the complete dataset
        ds_groupby_tas[f"wet_hour_mean_intensity_{range_name}"] = bin_values(days_within_range, "wet_hour_mean_intensity", groupby_var, "mean")
        ds_groupby_tas[f"wet_hour_max_intensity_{range_name}"]  = bin_values(days_within_range, "wet_hour_max_intensity",  groupby_var, "mean")
        ds_groupby_tas[f"wet_hour_count_{range_name}"]          = bin_values(days_within_range, "wet_hour_count",          groupby_var, "mean")
        ds_groupby_tas[f"pr_range_count_{range_name}"]          = bin_values(days_within_range, "pr",                      groupby_var, "count")
        ds_groupby_tas[f"pr_onset_time_{range_name}"]           = bin_values(days_within_range, "pr_onset_time",           groupby_var, "mean")

    # change temperature bins to values instead of intervals
    ds_groupby_tas["temperature_bin"] = [interval.left for interval in ds_groupby_tas.temperature_bin.values]

    return ds_groupby_tas

In [14]:
def make_all_datasets_from_hourly_data(ds_hourly, dataset_name, metadata, savefiles=True, period='eval', season='JJA'):
    """
    This function first takes simple hourly timeseries of temperature and precipitation, and calculates derived quantities on daily timescales.
    If files with these derived quantities don't exist yet, they are created and saved.
    These are then used to create three datasets:
        - high percentiles of hourly precipitation vs. mean daily temperature, count of all hours, count of wet hours
        - high percentiles of daily precipitation vs. mean daily temperature, count of all days, count of wet days
        - various precipitation characteristics: mean daily precipitation, count of days in each category, mean and maximum of wet hour intensities,
            time of onset of precipitation, and all these characteristics separated by various ranges of mean daily precipitation
    These three datasets are saved as three separate files.

    Args:
        ds_hourly (xr.Dataset): input dataset which contains hourly precipitation and temperature timeseries plus some metadata.
        dataset_name (str): One of 'kmscale', 'driving', 'GeoSphere'. Explains what type of dataset is being processed and is used in the output file name.
        metadata (xr.Dataset): metadata about a location: elevation, latitude, longitude, station ID.
        savefiles (bool, optional): A flag determining whether the created datasets should be saved. Defaults to True.
        period (str, optional): Time period, either evaluation ('eval'), historical ('hist'), or future RCP8.5 ('rcp'). Defaults to 'eval'.
        season (str): one of 'DJF', 'MAM', 'JJA', 'SON', 'allseasons'. The last one means we consider the whole year. Defaults to 'JJA'.

    Returns:
        N/A. It saves the produced results in files.
    """

    # create full names (paths) for extended timeseries which will hold quantities derived from hourly precipitation and temperature data
    extended_daily_file_out = f"{folder_processed_data}/extended_daily_measures_{dataset_name}_{period}_{season}.nc"
    extended_hourly_file_out = f"{folder_processed_data}/extended_hourly_measures_{dataset_name}_{period}_{season}.nc"

    # Check if the extended files exist: if not, create them
    if not (os.path.exists(extended_daily_file_out) and os.path.exists(extended_hourly_file_out)):
        # get extended hourly and daily data from simple hourly data
        ds_hourly_extended, ds_daily_extended = get_daily_stats(ds_hourly)
        ds_daily_extended["time"] = pd.to_datetime(ds_daily_extended["time"])
        print(f"Daily data obtained: {dataset_name} {period} {season}")

        # make datasets with extended daily statistics grouped by temperature bins
        if savefiles:
            ds_daily_extended.to_netcdf(extended_daily_file_out)
            ds_hourly_extended.to_netcdf(extended_hourly_file_out)
    # if files already exist, open them instead
    else:
        print("Daily data exists")
        ds_daily_extended = xr.open_dataset(extended_daily_file_out)
        ds_hourly_extended = xr.open_dataset(extended_hourly_file_out)

    # define dataset desctiptions
    description_dict = {
        "hdmean": "High percentiles of hourly precipitation vs. mean daily temperature, count of all hours, count of wet hours. Bins of 1°C, overlapping by 0.5°C.",
        "ddmean": "High percentiles of daily precipitation vs. mean daily temperature, count of all days, count of wet days. Bins of 1°C, overlapping by 0.5°C.",
    }

    # make datasets with hourly/daily quantiles and all/wet hour/day counts based on hourly/daily data
    ds_hdmean = make_quantiles_file(ds_hourly_extended, metadata, groupby_var="mean_daily_tas", wet_intensity=WET_HOUR_INTENSITY, description=description_dict["hdmean"], dataset_name=dataset_name)
    ds_ddmean = make_quantiles_file(ds_daily_extended, metadata, groupby_var="mean_daily_tas", wet_intensity=WET_DAY_INTENSITY, description=description_dict["ddmean"], dataset_name=dataset_name)

    # make datasets with various daily stats based on extended daily data
    ds_daily_stats_dmean = make_complete_daily_file(ds_daily_extended, metadata, groupby_var="mean_daily_tas", dataset_name=dataset_name)

    # save files only if the flag is True
    if savefiles:
        # fill in the blanks in the names based on which dataset is processed
        ds_hdmean.to_netcdf(f"{folder_processed_data}/quantiles_{dataset_name}_{period}_hdmean_{season}.nc")
        ds_ddmean.to_netcdf(f"{folder_processed_data}/quantiles_{dataset_name}_{period}_ddmean_{season}.nc")
        ds_daily_stats_dmean.to_netcdf(f"{folder_processed_data}/various_daily_stats_{dataset_name}_{period}_dmean_{season}.nc")

    return

In [14]:
def make_all_datasets_from_hourly_data_parallel(file_path, season_months, season, metadata, savefiles=True, period='eval'):
    """
    This functions serves as a parallelization wrapper of the 'make_all_datasets_from_hourly_data' function.
    It automatically determines the type of dataset (km-scale, driving, station data).
    If specified, this function also selects one season from the dataset before further processing.

    Args:
        file_path (str): path to the input file which contains precipitation and temperature timeseries plus some metadata.
            All the input files are in the .nc format.
        season_months (list or None): one of [12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11], None. If None, no selection is made, i.e. we take the whole year.
        season (str): one of 'DJF', 'MAM', 'JJA', 'SON', 'allseasons'. The last one means we consider the whole year.
        metadata (xr.Dataset): metadata about a location: elevation, latitude, longitude, station ID.
        savefiles (bool, optional): A flag determining whether the created datasets should be saved. Defaults to True.
        period (str, optional): Time period, either evaluation ('eval'), historical ('hist'), or future RCP8.5 ('rcp'). Defaults to 'eval'.

    Returns:
        N/A. It saves the produced results in files.
    """

    # determine which model we're dealing with based on the filepath
    model = file_path.split("/")[-2]

    # determine resolution based on the filepath
    if "ALP-3" in file_path:
        res = "kmscale"
        dataset_name = f"{res}_{model}"
    elif "EUR-12" in file_path:
        res = "driving"
        dataset_name = f"{res}_{model}"
    elif "Geo" in file_path or "geosphere" in file_path:
        dataset_name = "GeoSphere"
    
    # print what's data is being processed now
    print(f'{season} {dataset_name} {period}')

    # open file
    ds_model = xr.open_dataset(file_path)

    # subselect months if we don't do the whole year
    # It would have been smarter to make the extended timeseries only once and select seasons afterwards, but oh well...
    if season_months is not None:
        ds_model = ds_model.sel(time=ds_model.time.dt.month.isin(season_months), drop=True)
    
    # pass all the determined arguments further and create resulting datasets
    make_all_datasets_from_hourly_data(ds_model, dataset_name, metadata, savefiles=savefiles, period=period, season=season)
    
    return

# Make files

## Files per model

In [ ]:
# GeoSphere (station data)
# for some reason this doesn't work in parallel:the JupyterHub server crashes after every season and has to be restarted.
# for season_months, season in zip(months, seasons):
#     make_all_datasets_from_hourly_data_parallel(filepath_geosphere, season_months, season, metadata, savefiles=savefiles, period="eval")

# if needed, restart the server after every season and comment out the processed line
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[0], seasons[0], metadata, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[1], seasons[1], metadata, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[2], seasons[2], metadata, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[3], seasons[3], metadata, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[4], seasons[4], metadata, savefiles=savefiles, period="eval")

In [ ]:
# Model data
# get a list of files and corresponding time periods to be passed further
model_period_pairs = [
    (files_kmscale_eval, "eval"),
    (files_driving_eval, "eval"),
    (files_kmscale_hist, "hist"),
    (files_driving_hist, "hist"),
    (files_kmscale_rcp, "rcp"),
    (files_driving_rcp, "rcp"),        
    ]

# process data
with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as exe:
    for season_months, season in zip(months, seasons):
        for files, period in model_period_pairs:
            for file_in in files:
                result = exe.submit(make_all_datasets_from_hourly_data_parallel, file_in, season_months, season, metadata, savefiles=savefiles, period=period)

## Create ensemble files

In [ ]:
# create ensemble files: quantiles etc
for resolution in ["kmscale", "driving"]:
    for season in seasons:
        print(resolution, season)
        for suffix in ["hdmean", "ddmean",]:
            for period in ["eval", "hist", "rcp"]:
                
                print(suffix, period)
                # get list of files
                quantiles_files = sorted(glob.glob(f"{folder_processed_data}/quantiles_{resolution}*_{period}_{suffix}_{season}.nc"))
                # get names of models from the filenames
                models = [f.split("/")[-1].split(f"_{suffix}_{season}.nc")[0].split(f"{resolution}_")[-1] for f in quantiles_files]
                # open as one dataset
                ds_combined = xr.open_mfdataset(quantiles_files, concat_dim="model", combine="nested", coords="minimal", data_vars="different")
                # assign name to dimension
                ds_combined["model"] = models
                # save file
                ds_combined.to_netcdf(f"{folder_processed_data}/quantiles_{resolution}_ensemble_{period}_{suffix}_{season}.nc")

        # create ensemble files: various daily stats
        for suffix in ["dmean", ]:
            for period in ["eval", "hist", "rcp"]:
                print(suffix, period)
                # get list of files
                various_daily_stats_files = sorted(glob.glob(f"{folder_processed_data}/various_daily_stats*{resolution}*{period}_{suffix}_{season}.nc"))
                # get names of models from the filenames
                models = [f.split("/")[-1].split(f"_{suffix}_{season}.nc")[0].split(f"{resolution}_")[-1] for f in various_daily_stats_files]
                # open as one dataset
                ds_combined = xr.open_mfdataset(various_daily_stats_files, concat_dim="model", combine="nested", coords="minimal", data_vars="different")
                # assign name to dimension
                ds_combined["model"] = models
                # save file
                ds_combined.to_netcdf(f"{folder_processed_data}/various_daily_stats_{resolution}_ensemble_{period}_{suffix}_{season}.nc")
        print()